In [ ]:
from pyspark.sql import SparkSession
 
# Crear una sesión de Spark
spark = SparkSession.builder \
    .appName("Scrap NYCTaxi") \
    .getOrCreate()

In [ ]:
from notebookutils import mssparkutils

RUTA_RAW = "abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/raw/"

print("Iniciando barrido de limpieza en capa RAW...")

# Obtenemos todas las carpetas de años:
carpetas_anio = mssparkutils.fs.ls(RUTA_RAW)

carpetas_tipo_borradas = 0
carpetas_mes_borradas = 0

for c_anio in carpetas_anio:
    carpetas_mes = mssparkutils.fs.ls(c_anio.path)
    
    for c_mes in carpetas_mes:
        # Limpieza de carpetas de TIPO (Yellow, Green, etc.):
        sub_carpetas_tipo = mssparkutils.fs.ls(c_mes.path)
        
        for c_tipo in sub_carpetas_tipo:
            archivos = mssparkutils.fs.ls(c_tipo.path)
            
            # Si la carpeta está vacía o el archivo pesa 0 bytes, borramos la carpeta del tipo:
            if len(archivos) == 0 or archivos[0].size == 0:
                mssparkutils.fs.rm(c_tipo.path, True)
                carpetas_tipo_borradas += 1
        
        # Verificación del MES
        # Consultamos de nuevo el contenido del mes tras la limpieza anterior:
        contenido_mes_actualizado = mssparkutils.fs.ls(c_mes.path)
        
        if len(contenido_mes_actualizado) == 0:
            mssparkutils.fs.rm(c_mes.path, True)
            carpetas_mes_borradas += 1

print("---")
print(f"Limpieza finalizada.")
print(f"Subcarpetas de tipo eliminadas: {carpetas_tipo_borradas}")
print(f"Carpetas de mes eliminadas: {carpetas_mes_borradas}")

In [ ]:
spark.stop()